In [1]:
import numpy as np
import pandas as pd
from pandas.tseries.holiday import USFederalHolidayCalendar
df = pd.read_csv("../data/flights_prepared_final.csv")


In [2]:
df = df.dropna(subset=["ArrDel15"]).reset_index(drop=True)
df["ArrDel15"] = df["ArrDel15"].astype("int8")
for c in ["Origin", "Dest", "IATA_Code_Operating_Airline"]:
    df[c] = df[c].astype("category")
print(df.shape)
 

(7917259, 26)


In [3]:
df["dep_hour"] = (df["CRSDepTime"] // 100).clip(upper=23)
df["arr_hour"] = (df["CRSArrTime"] // 100).clip(upper=23)
df["dep_minutes"] = df["dep_hour"] * 60 + (df["CRSDepTime"] % 100).clip(upper=59)
df["arr_minutes"] = df["arr_hour"] * 60 + (df["CRSArrTime"] % 100).clip(upper=59)
 
# le vol arrive-t-il après minuit ?
df["is_overnight"] = (df["arr_minutes"] < df["dep_minutes"]).astype("int8")
 
# sin/cos : pour que 23h59 et 00h01 soient "proches" (utile surtout pour un modèle linéaire)
angle = 2 * np.pi * df["dep_minutes"] / 1440
df["dep_sin"] = np.sin(angle).astype("float32")
df["dep_cos"] = np.cos(angle).astype("float32")
 
 

In [5]:
 #Seul le samedi (6) se distingue dans l'EDA : une seule colonne suffit
df["is_saturday"] = (df["DayOfWeek"] == 6).astype("int8")
 
# même idée que pour l'heure : décembre et janvier sont voisins
angle = 2 * np.pi * df["Month"] / 12
df["Month_sin"] = np.sin(angle).astype("float32")
df["Month_cos"] = np.cos(angle).astype("float32")
 

In [6]:
df["avg_speed"] = (df["Distance"] / df["CRSElapsedTime"].clip(lower=1)).astype("float32")

In [8]:
df["FlightDate"] = pd.to_datetime(df["FlightDate"])
fetes = USFederalHolidayCalendar().holidays(
    df["FlightDate"].min() - pd.Timedelta(days=30),
    df["FlightDate"].max() + pd.Timedelta(days=30)).values
 
dates = np.sort(df["FlightDate"].unique())
i = np.searchsorted(fetes, dates)
avant = fetes[np.clip(i - 1, 0, len(fetes) - 1)]
apres = fetes[np.clip(i, 0, len(fetes) - 1)]
jours = np.minimum(
    np.abs((dates - avant).astype("timedelta64[D]").astype(int)),
    np.abs((apres - dates).astype("timedelta64[D]").astype(int)))
 
df["days_to_holiday"] = df["FlightDate"].map(pd.Series(jours, index=pd.DatetimeIndex(dates))).astype("int16")
df["near_holiday"] = (df["days_to_holiday"] <= 2).astype("int8")

In [9]:
def compter(cles, col):
    return df.groupby(cles, observed=True)[col].transform("size").astype("int16")
 
df["origin_dep_hour_count"] = compter(["Origin", "FlightDate", "dep_hour"], "dep_minutes")   # même aéroport, même heure
df["origin_dep_day_count"] = compter(["Origin", "FlightDate"], "dep_minutes")                # même aéroport, même jour
df["dest_arr_hour_count"] = compter(["Dest", "FlightDate", "arr_hour"], "arr_minutes")       # arrivées à destination
df["airline_day_count"] = compter(["IATA_Code_Operating_Airline", "FlightDate"], "dep_minutes")
 
# combien de vols partent AVANT celui-ci le même jour au même aéroport
# (les retards s'accumulent au fil de la journée)
rang = df.groupby(["Origin", "FlightDate"], observed=True)["dep_minutes"].rank(method="min")
df["origin_rank_day"] = (rang - 1).astype("int16")

In [10]:
MODEL_FEATURES = [
    "dep_minutes", "arr_minutes", "dep_sin", "dep_cos", "is_overnight",
    "is_saturday", "Month_sin", "Month_cos",
    "Distance", "avg_speed", "is_codeshare",
    "days_to_holiday", "near_holiday",
    "origin_dep_hour_count", "origin_dep_day_count", "dest_arr_hour_count",
    "airline_day_count", "origin_rank_day",
]
 
# Colonnes gardées SEULEMENT pour expliquer les résultats (pas pour le modèle)
EXPLAIN_COLS = [c for c in [
    "Origin", "Dest", "IATA_Code_Marketing_Airline", "IATA_Code_Operating_Airline",
    "Flight_Number_Marketing_Airline", "OriginState", "DestState",
    "Origin_encoded", "Dest_encoded", "IATA_Code_Marketing_Airline_encoded",
    "IATA_Code_Operating_Airline_encoded", "OriginState_encoded", "DestState_encoded",
] if c in df.columns]
 
# FlightDate est gardée pour faire le split train/val/test avec le modèle (ne pas l'utiliser comme variable)
out = df[["FlightDate", "ArrDel15"] + MODEL_FEATURES + EXPLAIN_COLS]
print(out.shape, "| valeurs manquantes dans les features :", int(out[MODEL_FEATURES].isna().sum().sum()))
 
out.to_csv("../data/flights_features.csv", index=False)

(7917259, 33) | valeurs manquantes dans les features : 0
